<a href="https://colab.research.google.com/github/cpython-projects/da_27_07_2026/blob/main/lesson_18_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Датасет:** [House Sales in King County, USA (Kaggle)](https://www.kaggle.com/datasets/harlfoxem/housesalesprediction)
Файл: `kc_house_data.csv`

**!** Не рахуйте середнє одразу. Розподіл цін у нашому датасеті має сильний правий перекіс (high skewness) — довгий хвіст із наддорогих об'єктів. Якщо рахувати середнє «в лоб», ви отримаєте цифри, які не існують у реальності для 90% населення.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

url = 'https://raw.githubusercontent.com/cpython-projects/python_da_17_03_26/refs/heads/main/kc_house_data.csv'
df = pd.read_csv(url)
df['date'] = pd.to_datetime(df['date'], format='%Y%m%dT%H%M%S')

In [2]:
df

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,2014-10-13,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180.0,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,2014-12-09,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170.0,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,2015-02-25,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770.0,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,2014-12-09,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050.0,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,2015-02-18,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680.0,0,1987,0,98074,47.6168,-122.045,1800,7503
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,263000018,2014-05-21,360000.0,3,2.50,1530,1131,3.0,0,0,...,8,1530.0,0,2009,0,98103,47.6993,-122.346,1530,1509
21609,6600060120,2015-02-23,400000.0,4,2.50,2310,5813,2.0,0,0,...,8,2310.0,0,2014,0,98146,47.5107,-122.362,1830,7200
21610,1523300141,2014-06-23,402101.0,2,0.75,1020,1350,2.0,0,0,...,7,1020.0,0,2009,0,98144,47.5944,-122.299,1020,2007
21611,291310100,2015-01-16,400000.0,3,2.50,1600,2388,2.0,0,0,...,8,1600.0,0,2004,0,98027,47.5345,-122.069,1410,1287


## Task 1
Потрібно зрозуміти, чи дійсно кількість спалень прямо пропорційна вартості житла.
Порахуй, будь ласка, середню ціну будинків залежно від кількості спалень. Нам треба виділити ті категорії, де нерухомість коштує аномально дорого або дешево для свого розміру».

In [3]:
df.columns.to_list()

['id',
 'date',
 'price',
 'bedrooms',
 'bathrooms',
 'sqft_living',
 'sqft_lot',
 'floors',
 'waterfront',
 'view',
 'condition',
 'grade',
 'sqft_above',
 'sqft_basement',
 'yr_built',
 'yr_renovated',
 'zipcode',
 'lat',
 'long',
 'sqft_living15',
 'sqft_lot15']

In [9]:
task_1 = df.groupby('bedrooms').agg(
    avg_price=('price', 'mean'),
    avg_sqft_living=('sqft_living', 'mean'),
    num=('price', 'size')
).reset_index()
task_1['avg_price_sqft'] = task_1.avg_price / task_1.avg_sqft_living
task_1.round(1)
task_1 = task_1[task_1.num > 35]
px.bar(task_1, x='bedrooms', y='avg_price_sqft').show()

In [11]:
task_1 = df.groupby('bedrooms').agg(
    median_price=('price', 'median'),
    median_sqft_living=('sqft_living', 'median'),
    num=('price', 'size')
).reset_index()
task_1['median_price_sqft'] = task_1.median_price / task_1.median_sqft_living
task_1.round(1)
task_1 = task_1[task_1.num > 35]
px.bar(task_1, x='bedrooms', y='median_price_sqft').show()


## Task 2
Ми готуємо звіт по якості забудови (grade). Для кожного рівня якості мені потрібна повна картина: скільки всього таких об'єктів у базі, яка мінімальна та максимальна ціна в кожній групі».

In [12]:
task_2 = df.groupby('grade').agg(
    min_price=('price', 'min'),
    max_price=('price', 'max'),
    num=('price', 'size')
).reset_index()
task_2

,grade,min_price,max_price,num
0,1,142000.0,142000.0,1
1,3,75000.0,280000.0,3
2,4,80000.0,435000.0,29
3,5,78000.0,795000.0,242
4,6,82000.0,1200000.0,2038
5,7,90000.0,2050000.0,8981
6,8,140000.0,3070000.0,6068
7,9,230000.0,2700000.0,2615
8,10,316000.0,3600000.0,1134
9,11,420000.0,7062500.0,399


## Task 3
Інвестори питають, як змінювався ринок протягом року.

In [15]:
task_3 = df.set_index('date').resample('ME').agg(
    median_price=('price', 'median'),
    deals=('price', 'size')
).reset_index().round(1)

task_3

,date,median_price,deals
0,2014-05-31,465000.0,1768
1,2014-06-30,465000.0,2180
2,2014-07-31,465000.0,2211
3,2014-08-31,442100.0,1940
4,2014-09-30,450000.0,1774
5,2014-10-31,446900.0,1878
6,2014-11-30,435000.0,1411
7,2014-12-31,432500.0,1471
8,2015-01-31,438500.0,978
9,2015-02-28,425545.0,1250


In [16]:
px.line(task_3, x='date', y='median_price').show()

In [17]:
px.bar(task_3, x='date', y='deals').show()

## Task 4
Ми хочемо розробити маркетингову стратегію для трьох типів клієнтів: тих, хто шукає "історичні" будинки (до 1950 року), "класику" (1950–2000) та "новобудови" (після 2000-го).
Розбий всі будинки на ці три групи за роком будівництва та скажи, у якій групі зараз найвищий середній показник стану».


In [18]:
df.columns.to_list()

['id',
 'date',
 'price',
 'bedrooms',
 'bathrooms',
 'sqft_living',
 'sqft_lot',
 'floors',
 'waterfront',
 'view',
 'condition',
 'grade',
 'sqft_above',
 'sqft_basement',
 'yr_built',
 'yr_renovated',
 'zipcode',
 'lat',
 'long',
 'sqft_living15',
 'sqft_lot15']

In [19]:
bins = [df.yr_built.min() - 1, 1950, 2000, df.yr_built.max()]
labels = ['hist', 'classic', 'new']
df['year_class'] = pd.cut(df.yr_built, bins=bins, labels=labels)
df

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,year_class
0,7129300520,2014-10-13,221900.0,3,1.00,1180,5650,1.0,0,0,...,1180.0,0,1955,0,98178,47.5112,-122.257,1340,5650,classic
1,6414100192,2014-12-09,538000.0,3,2.25,2570,7242,2.0,0,0,...,2170.0,400,1951,1991,98125,47.7210,-122.319,1690,7639,classic
2,5631500400,2015-02-25,180000.0,2,1.00,770,10000,1.0,0,0,...,770.0,0,1933,0,98028,47.7379,-122.233,2720,8062,hist
3,2487200875,2014-12-09,604000.0,4,3.00,1960,5000,1.0,0,0,...,1050.0,910,1965,0,98136,47.5208,-122.393,1360,5000,classic
4,1954400510,2015-02-18,510000.0,3,2.00,1680,8080,1.0,0,0,...,1680.0,0,1987,0,98074,47.6168,-122.045,1800,7503,classic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,263000018,2014-05-21,360000.0,3,2.50,1530,1131,3.0,0,0,...,1530.0,0,2009,0,98103,47.6993,-122.346,1530,1509,new
21609,6600060120,2015-02-23,400000.0,4,2.50,2310,5813,2.0,0,0,...,2310.0,0,2014,0,98146,47.5107,-122.362,1830,7200,new
21610,1523300141,2014-06-23,402101.0,2,0.75,1020,1350,2.0,0,0,...,1020.0,0,2009,0,98144,47.5944,-122.299,1020,2007,new
21611,291310100,2015-01-16,400000.0,3,2.50,1600,2388,2.0,0,0,...,1600.0,0,2004,0,98027,47.5345,-122.069,1410,1287,new


In [20]:
task_4 = df.groupby('year_class', observed=True)['condition'].agg(
    ['mean', 'count']
).reset_index().round(1)
task_4

,year_class,mean,count
0,hist,3.6,5187
1,classic,3.5,11885
2,new,3.0,4541


## Task 5
Відділ продажів хоче розділити всі об'єкти на 5 рівних за кількістю груп — від "бюджетних" до "люкс-сегменту" (по 20% будинків у кожній).
Нам потрібно порівняти ці групи: яка середня площа житла (sqft_living) у найдешевшому сегменті порівняно з найдорожчим?»

In [22]:
df['price_class'] = pd.qcut(df.price, q=5, labels=['Б 0', 'Б 1', 'Норм', 'Бізнес', 'Люкс'])
df

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,year_class,price_class
0,7129300520,2014-10-13,221900.0,3,1.00,1180,5650,1.0,0,0,...,0,1955,0,98178,47.5112,-122.257,1340,5650,classic,Б 0
1,6414100192,2014-12-09,538000.0,3,2.25,2570,7242,2.0,0,0,...,400,1951,1991,98125,47.7210,-122.319,1690,7639,classic,Бізнес
2,5631500400,2015-02-25,180000.0,2,1.00,770,10000,1.0,0,0,...,0,1933,0,98028,47.7379,-122.233,2720,8062,hist,Б 0
3,2487200875,2014-12-09,604000.0,4,3.00,1960,5000,1.0,0,0,...,910,1965,0,98136,47.5208,-122.393,1360,5000,classic,Бізнес
4,1954400510,2015-02-18,510000.0,3,2.00,1680,8080,1.0,0,0,...,0,1987,0,98074,47.6168,-122.045,1800,7503,classic,Норм
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,263000018,2014-05-21,360000.0,3,2.50,1530,1131,3.0,0,0,...,0,2009,0,98103,47.6993,-122.346,1530,1509,new,Б 1
21609,6600060120,2015-02-23,400000.0,4,2.50,2310,5813,2.0,0,0,...,0,2014,0,98146,47.5107,-122.362,1830,7200,new,Норм
21610,1523300141,2014-06-23,402101.0,2,0.75,1020,1350,2.0,0,0,...,0,2009,0,98144,47.5944,-122.299,1020,2007,new,Норм
21611,291310100,2015-01-16,400000.0,3,2.50,1600,2388,2.0,0,0,...,0,2004,0,98027,47.5345,-122.069,1410,1287,new,Норм


In [25]:
task_5 = df.groupby('price_class', observed=True).agg(
    avg_price=('price', 'mean'),
    median_price=('price', 'median'),
    num=('price', 'size')
).reset_index().round(1)
task_5

,price_class,avg_price,median_price,num
0,Б 0,237132.3,245000.0,4323
1,Б 1,346221.9,345000.0,4325
2,Норм,453949.9,450000.0,4329
3,Бізнес,599529.0,595000.0,4313
4,Люкс,1063954.4,887000.0,4323


## Task 6
Я хочу побачити зведену таблицю, де в рядках буде оцінка виду (view), а в стовпцях — наявність виходу до води (waterfront).
У комірках має бути середня ціна за квадратний метр. Це допоможе нам зрозуміти, за що саме переплачують клієнти: за краєвид чи за безпосередній доступ до водойми».

In [26]:
df['price_per_sqft'] = df.price / df.sqft_living
df

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,year_class,price_class,price_per_sqft
0,7129300520,2014-10-13,221900.0,3,1.00,1180,5650,1.0,0,0,...,1955,0,98178,47.5112,-122.257,1340,5650,classic,Б 0,188.050847
1,6414100192,2014-12-09,538000.0,3,2.25,2570,7242,2.0,0,0,...,1951,1991,98125,47.7210,-122.319,1690,7639,classic,Бізнес,209.338521
2,5631500400,2015-02-25,180000.0,2,1.00,770,10000,1.0,0,0,...,1933,0,98028,47.7379,-122.233,2720,8062,hist,Б 0,233.766234
3,2487200875,2014-12-09,604000.0,4,3.00,1960,5000,1.0,0,0,...,1965,0,98136,47.5208,-122.393,1360,5000,classic,Бізнес,308.163265
4,1954400510,2015-02-18,510000.0,3,2.00,1680,8080,1.0,0,0,...,1987,0,98074,47.6168,-122.045,1800,7503,classic,Норм,303.571429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21608,263000018,2014-05-21,360000.0,3,2.50,1530,1131,3.0,0,0,...,2009,0,98103,47.6993,-122.346,1530,1509,new,Б 1,235.294118
21609,6600060120,2015-02-23,400000.0,4,2.50,2310,5813,2.0,0,0,...,2014,0,98146,47.5107,-122.362,1830,7200,new,Норм,173.160173
21610,1523300141,2014-06-23,402101.0,2,0.75,1020,1350,2.0,0,0,...,2009,0,98144,47.5944,-122.299,1020,2007,new,Норм,394.216667
21611,291310100,2015-01-16,400000.0,3,2.50,1600,2388,2.0,0,0,...,2004,0,98027,47.5345,-122.069,1410,1287,new,Норм,250.000000


In [27]:
task_6 = df.pivot_table(
    index='view',
    columns='waterfront',
    aggfunc='median',
    values='price_per_sqft'
).round(1)
task_6

waterfront,0,1
view,,
0,239.3,NaN
1,297.3,327.3
2,283.4,486.2
3,295.8,505.6
4,356.6,532.3
